In [14]:
%%writefile generate_data.py
import numpy as np
import pandas as pd

# =====================================================
# GENERATOR 1: DUKA ELECTRONICS SALES DATA
# Random seed: 2026
# =====================================================

rng = np.random.default_rng(2026)

n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),

    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(
                      rng.integers(0, 180, n), unit="D"
                  ),

    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),

    "product": rng.choice(list(products), n),

    "quantity": rng.integers(1, 6, n),

    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)


# =====================================================
# GENERATOR 2: MOBILE MONEY AGENT NETWORK
# Random seed: 44
# =====================================================

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],

    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"],
        20
    ),

    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

    "agent_id": rng.choice(agents["agent_id"], n),

    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"],
        n
    ),

    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),

    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

# Inject missing, invalid and suspicious amounts
idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

# Inject duplicate transactions
txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

# Inject unknown agents
txns.loc[
    rng.choice(len(txns), 8, replace=False), "agent_id"
] = "AG099"

# Save generated datasets
agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)


print("ALL DATASETS GENERATED SUCCESSFULLY")
print("-" * 45)

print("Duka sales records:", len(sales))
print("Agent records:", len(agents))
print("Agent transaction records:", len(txns))


Writing generate_data.py


In [15]:
!python generate_data.py

ALL DATASETS GENERATED SUCCESSFULLY
---------------------------------------------
Duka sales records: 1000
Agent records: 20
Agent transaction records: 612


In [16]:
import os

print(os.listdir())

['.config', 'duka_branch_summary.xlsx', 'duka_sales.csv', 'agent_transactions.csv', 'generate_data.py', '.ipynb_checkpoints', 'agents.csv', 'excel', 'sample_data']


# Duka Electronics Sales Exploration

**Project:** PLP Data Analysis – Assignment 3

**Client:** Duka Electronics

**Reporting Period:** January – June 2026

**Objective:** Analyse sales performance across five branches, identify revenue trends, evaluate product performance and provide actionable business recommendations.


## Executive Summary

- Duka Electronics recorded **1,000 orders** across five branches between January and June 2026.
- Nairobi was the best-performing branch, generating **KES 26,106,100** from 376 orders.
- Laptops generated the highest product revenue of **KES 41,535,000**, while tablets recorded the highest sales volume at 691 units.
- Weekdays generated **KES 51,856,200**, compared with KES 19,936,300 on weekends.
- April was the best-performing month with **KES 13,518,300**, approximately 23.98% higher than June, the weakest month.

In [17]:
# Import required libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [18]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

print("Duka dataset regenerated:", len(sales), "records")

Duka dataset regenerated: 1000 records


In [19]:
# Load Duka Electronics sales data

df = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

print("Duka sales dataset loaded successfully!")

Duka sales dataset loaded successfully!


In [20]:
print("FIRST FIVE SALES RECORDS")
print("-" * 50)

display(df.head())

FIRST FIVE SALES RECORDS
--------------------------------------------------


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [21]:
print("DATASET SHAPE")
print("-" * 50)

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

DATASET SHAPE
--------------------------------------------------
Number of rows: 1000
Number of columns: 7


In [22]:
print("DATASET INFORMATION")
print("-" * 50)

df.info()

DATASET INFORMATION
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [23]:
print("MISSING VALUES")
print("-" * 50)

print(df.isnull().sum())

MISSING VALUES
--------------------------------------------------
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64


In [24]:
print("ORDER DATE RANGE")
print("-" * 50)

print("Earliest order:", df["order_date"].min())
print("Latest order:", df["order_date"].max())

ORDER DATE RANGE
--------------------------------------------------
Earliest order: 2026-01-01 00:00:00
Latest order: 2026-06-29 00:00:00


In [25]:
print("KEY CATEGORIES")
print("-" * 50)

print("\nBranches:")
print(df["branch"].value_counts())

print("\nProducts:")
print(df["product"].value_counts())

print("\nSales channels:")
print(df["channel"].value_counts())

KEY CATEGORIES
--------------------------------------------------

Branches:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

Products:
product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

Sales channels:
channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64


## First-Five-Minutes Inspection

The Duka Electronics dataset contains 1,000 sales orders and 7 original columns. The dataset covers the first six months of 2026, from January to June. The records represent five branches: Nairobi, Mombasa, Kisumu, Nakuru and Eldoret. Products include Phones, Laptops, Headphones, Tablets and Chargers, while sales occur through In-store, Online and Phone order channels.

The initial inspection checks the dataset structure, column data types, missing values, date range and distribution of key categories before conducting further analysis.

## Task 1: Create Derived Sales Columns

Five additional columns are created to support sales analysis:
- revenue: total value of each order.
- month: month in which the order was placed.
- weekday: day of the week on which the order was placed.
- is_weekend: identifies Saturday and Sunday orders.
- price_band: classifies products as Premium or Value.

In [26]:
# 1. Calculate total revenue for each order
df["revenue"] = df["quantity"] * df["unit_price"]

# 2. Extract the month name
df["month"] = df["order_date"].dt.month_name()

# 3. Extract the weekday name
df["weekday"] = df["order_date"].dt.day_name()

# 4. Identify weekend orders
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5

# 5. Classify products by price
df["price_band"] = np.where(
    df["unit_price"] >= 28000,
    "Premium",
    "Value"
)

# Display the updated dataset
print("UPDATED DUKA SALES DATA")
print("-" * 50)

display(df.head(10))

print("\nUpdated dataset shape:", df.shape)

UPDATED DUKA SALES DATA
--------------------------------------------------


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value
5,1006,2026-03-26,Nairobi,Headphones,1,Online,2500,2500,March,Thursday,False,Value
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
7,1008,2026-03-08,Nairobi,Charger,5,In-store,1200,6000,March,Sunday,True,Value
8,1009,2026-04-26,Mombasa,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
9,1010,2026-03-05,Nakuru,Charger,5,Online,1200,6000,March,Thursday,False,Value



Updated dataset shape: (1000, 12)


In [27]:
print("DERIVED COLUMN VERIFICATION")
print("-" * 50)

print("\nRevenue summary:")
print(df["revenue"].describe())

print("\nPrice band distribution:")
print(df["price_band"].value_counts())

print("\nWeekend distribution:")
print(df["is_weekend"].value_counts())

print("\nMonth distribution:")
print(df["month"].value_counts())

DERIVED COLUMN VERIFICATION
--------------------------------------------------

Revenue summary:
count      1000.00000
mean      71792.50000
std       84180.91827
min        1200.00000
25%        6000.00000
50%       45000.00000
75%      112000.00000
max      325000.00000
Name: revenue, dtype: float64

Price band distribution:
price_band
Value      565
Premium    435
Name: count, dtype: int64

Weekend distribution:
is_weekend
False    723
True     277
Name: count, dtype: int64

Month distribution:
month
March       186
January     173
April       167
June        162
February    159
May         153
Name: count, dtype: int64


In [28]:
# Calculate total revenue and number of orders for each branch

branch_performance = df.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    number_of_orders=("order_id", "count")
)

# Rank branches by total revenue
revenue_ranking = branch_performance.sort_values(
    by="total_revenue",
    ascending=False
)

# Rank branches by number of orders
orders_ranking = branch_performance.sort_values(
    by="number_of_orders",
    ascending=False
)

print("BRANCH RANKING BY TOTAL REVENUE")
print("-" * 50)
display(revenue_ranking)

print("\nBRANCH RANKING BY NUMBER OF ORDERS")
print("-" * 50)
display(orders_ranking)

BRANCH RANKING BY TOTAL REVENUE
--------------------------------------------------


,total_revenue,number_of_orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117



BRANCH RANKING BY NUMBER OF ORDERS
--------------------------------------------------


,total_revenue,number_of_orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117


In [29]:
# Compare the revenue and order rankings

comparison = branch_performance.copy()

comparison["revenue_rank"] = comparison["total_revenue"].rank(
    ascending=False,
    method="min"
)

comparison["orders_rank"] = comparison["number_of_orders"].rank(
    ascending=False,
    method="min"
)

comparison = comparison.sort_values("revenue_rank")

print("BRANCH RANKING COMPARISON")
print("-" * 60)

display(comparison)

BRANCH RANKING COMPARISON
------------------------------------------------------------


,total_revenue,number_of_orders,revenue_rank,orders_rank
branch,,,,
Nairobi,26106100,376,1.0,1.0
Mombasa,14003700,186,2.0,2.0
Nakuru,11970400,162,3.0,3.0
Kisumu,10868100,159,4.0,4.0
Eldoret,8844200,117,5.0,5.0


### Branch Performance Analysis

Nairobi emerged as the best-performing branch, generating total revenue of **KES 26,106,100** from **376 orders**. Mombasa ranked second with KES 14,003,700 from 186 orders, while Nakuru ranked third with KES 11,970,400 from 162 orders. Kisumu and Eldoret ranked fourth and fifth, respectively.

The ranking by total revenue was identical to the ranking by number of orders. This indicates that branches with higher order volumes also generated higher total revenue. Eldoret recorded the lowest performance, suggesting a need to investigate opportunities to increase customer traffic and sales.

**Business recommendation:** Duka Electronics should investigate the factors driving Nairobi's strong performance and consider applying successful strategies to lower-performing branches, particularly Eldoret.

In [30]:
# Calculate product performance
product_performance = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum"),
    number_of_orders=("order_id", "count")
)

# Rank the top three products by revenue
top_products_revenue = product_performance.sort_values(
    by="total_revenue",
    ascending=False
).head(3)

# Rank the top three products by units sold
top_products_units = product_performance.sort_values(
    by="units_sold",
    ascending=False
).head(3)

print("TOP 3 PRODUCTS BY REVENUE")
display(top_products_revenue)

print("\nTOP 3 PRODUCTS BY UNITS SOLD")
display(top_products_units)

TOP 3 PRODUCTS BY REVENUE


,total_revenue,units_sold,number_of_orders
product,,,
Laptop,41535000,639,220
Tablet,19348000,691,215
Phone,8880000,592,194



TOP 3 PRODUCTS BY UNITS SOLD


,total_revenue,units_sold,number_of_orders
product,,,
Tablet,19348000,691,215
Laptop,41535000,639,220
Phone,8880000,592,194


### Product Performance Analysis

Laptops generated the highest revenue at **KES 41,535,000**, despite selling 639 units. Tablets ranked second in revenue at KES 19,348,000 but recorded the highest sales volume, with 691 units sold. Phones ranked third, generating KES 8,880,000 from 592 units.

The difference in rankings is mainly explained by product pricing. Laptops have a higher unit price of KES 65,000, allowing them to generate more revenue even with fewer units sold than tablets, which cost KES 28,000 each.

**Business recommendation:** Duka Electronics should maintain sufficient laptop stock because of their strong revenue contribution while continuing to promote tablets because of their high sales volume. The business should also investigate opportunities to increase phone sales.

In [31]:
# Compare weekend and weekday performance
weekend_comparison = df.groupby("is_weekend").agg(
    number_of_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

# Replace Boolean labels with readable names
weekend_comparison.index = weekend_comparison.index.map(
    {True: "Weekend", False: "Weekday"}
)

display(weekend_comparison)

,number_of_orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


### Weekend vs Weekday Sales Analysis

Weekdays recorded **723 orders**, generating total revenue of **KES 51,856,200**, while weekends recorded 277 orders and generated KES 19,936,300. This indicates that most sales activity occurs during weekdays.

However, the average order revenue was slightly higher on weekends at KES 71,972.20 compared with KES 71,723.65 on weekdays. This suggests that weekend customers spend marginally more per transaction, although fewer transactions take place.

**Business recommendation:** Duka Electronics should maintain sufficient staffing and stock during weekdays to handle higher sales volumes. The business could also introduce weekend promotions to attract more customers and increase weekend order volumes.

In [32]:
# Calculate total revenue for each month
monthly_revenue = df.groupby("month")["revenue"].sum()

# Arrange months in calendar order
monthly_revenue = monthly_revenue.reindex([
    "January", "February", "March",
    "April", "May", "June"
])

print("MONTHLY REVENUE")
display(monthly_revenue)

# Identify the best and worst months
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_value = monthly_revenue.max()
worst_value = monthly_revenue.min()

# Calculate percentage difference relative to the worst month
percentage_difference = (
    (best_value - worst_value) / worst_value
) * 100

print("Best-performing month:", best_month)
print("Best-month revenue: KES", best_value)

print("Worst-performing month:", worst_month)
print("Worst-month revenue: KES", worst_value)

print("Percentage difference:", round(percentage_difference, 2), "%")

MONTHLY REVENUE


,revenue
month,
January,11786600
February,11772000
March,12396800
April,13518300
May,11415400
June,10903400


Best-performing month: April
Best-month revenue: KES 13518300
Worst-performing month: June
Worst-month revenue: KES 10903400
Percentage difference: 23.98 %


### Monthly Revenue Performance Analysis

April was the best-performing month, generating total revenue of **KES 13,518,300**, while June was the worst-performing month, generating KES 10,903,400. April's revenue was approximately **23.98% higher** than June's revenue.

Revenue generally increased from January to April before declining in May and June. This suggests that sales performance weakened towards the end of the reporting period.

**Business recommendation:** Duka Electronics should investigate the factors that contributed to April's strong performance and identify the reasons for the decline in May and June. The business could use targeted promotions and stock planning to improve sales during weaker months.

In [33]:
# Filter Premium Online orders from Nairobi and Mombasa
premium_online = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Display the filtered records
display(premium_online.head(10))

# Calculate the required figures
print("Number of matching orders:", len(premium_online))

print(
    "Total revenue: KES",
    premium_online["revenue"].sum()
)

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium
48,1049,2026-03-25,Mombasa,Tablet,1,Online,28000,28000,March,Wednesday,False,Premium
58,1059,2026-04-30,Mombasa,Laptop,1,Online,65000,65000,April,Thursday,False,Premium
77,1078,2026-06-12,Nairobi,Tablet,2,Online,28000,56000,June,Friday,False,Premium
82,1083,2026-03-16,Nairobi,Tablet,3,Online,28000,84000,March,Monday,False,Premium
103,1104,2026-03-08,Mombasa,Laptop,3,Online,65000,195000,March,Sunday,True,Premium


Number of matching orders: 83
Total revenue: KES 10218000


In [34]:
# Create the outputs folder if it does not exist
import os

os.makedirs("outputs", exist_ok=True)

# Export the filtered data
premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("CSV file exported successfully!")

CSV file exported successfully!


In [35]:
exported_data = pd.read_csv(
    "outputs/premium_online_coast_capital.csv"
)

print("Exported file shape:", exported_data.shape)
display(exported_data.head())

Exported file shape: (83, 12)


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
1,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
2,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
3,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
4,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


### Premium Online Orders: Nairobi and Mombasa

The analysis identified **83 orders** matching all three conditions: Premium price band, Online sales channel, and either Nairobi or Mombasa branch. These orders generated total revenue of **KES 10,218,000**.

The filtered records were successfully exported to `outputs/premium_online_coast_capital.csv`. Verification confirmed that the exported file contains 83 rows and 12 columns.

**Business recommendation:** Duka Electronics should strengthen its online marketing of premium products in Nairobi and Mombasa. These results provide a useful starting point for evaluating premium online sales performance in the capital and coastal markets.

In [36]:
import os

# Create the excel folder
os.makedirs("excel", exist_ok=True)

# Export the analysed dataset
with pd.ExcelWriter(
    "excel/duka_branch_summary.xlsx",
    engine="openpyxl"
) as writer:
    df.to_excel(
        writer,
        sheet_name="Sales Data",
        index=False
    )

print("Excel workbook created successfully!")

Excel workbook created successfully!


In [37]:
import os

print(os.listdir("excel"))

['duka_branch_summary.xlsx']


In [ ]:
from google.colab import files

files.download("excel/duka_branch_summary.xlsx")

In [39]:
from google.colab import files
import os
import shutil

# Open a file picker on your computer
uploaded = files.upload()

# Create the required folder
os.makedirs("excel", exist_ok=True)

# Get the uploaded filename
filename = list(uploaded.keys())[0]

# Copy the workbook into the required location
shutil.copy2(
    filename,
    "excel/duka_branch_summary.xlsx"
)

print("SUCCESS! Workbook uploaded and saved.")
print("Location: excel/duka_branch_summary.xlsx")

IndexError: list index out of range

In [40]:
import pandas as pd

workbook = pd.ExcelFile("excel/duka_branch_summary.xlsx")

print("Workbook sheets:")
print(workbook.sheet_names)

Workbook sheets:
['Sales Data']


In [45]:
import os

required_files = [
    "generate_data.py",
    "duka_sales_exploration.ipynb",
    "cleaning_pipeline.py",
    "excel/duka_branch_summary.xlsx",
    "outputs/premium_online_coast_capital.csv"
]

for file in required_files:
    if os.path.exists(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

FOUND: generate_data.py
FOUND: duka_sales_exploration.ipynb
MISSING: cleaning_pipeline.py
FOUND: excel/duka_branch_summary.xlsx
FOUND: outputs/premium_online_coast_capital.csv


In [42]:
import os

print("CURRENT DIRECTORY:")
print(os.getcwd())

print("\nFILES AVAILABLE:")
print(os.listdir())

CURRENT DIRECTORY:
/content

FILES AVAILABLE:
['.config', 'duka_branch_summary.xlsx', 'duka_sales.csv', 'agent_transactions.csv', 'generate_data.py', '.ipynb_checkpoints', 'outputs', 'agents.csv', 'excel', 'sample_data']


In [43]:
print("DataFrame available:", "df" in globals())

if "df" in globals():
    print("Sales records:", len(df))
    print("Columns:", list(df.columns))

print("Filtered data available:", "premium_online" in globals())

DataFrame available: True
Sales records: 1000
Columns: ['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band']
Filtered data available: True


In [44]:
import os

print(os.path.exists("duka_sales_exploration.ipynb"))

True


In [46]:
import os

required_files = [
    "generate_data.py",
    "duka_sales_exploration.ipynb",
    "cleaning_pipeline.py",
    "excel/duka_branch_summary.xlsx",
    "outputs/premium_online_coast_capital.csv",
    "agent_cleanup.ipynb",
    "outputs/agent_transactions_clean.csv",
    "README.md",
    "requirements.txt"
]

for file in required_files:
    status = "FOUND" if os.path.exists(file) else "MISSING"
    print(f"{status}: {file}")

FOUND: generate_data.py
FOUND: duka_sales_exploration.ipynb
MISSING: cleaning_pipeline.py
FOUND: excel/duka_branch_summary.xlsx
FOUND: outputs/premium_online_coast_capital.csv
MISSING: agent_cleanup.ipynb
MISSING: outputs/agent_transactions_clean.csv
MISSING: README.md
MISSING: requirements.txt


In [47]:
%%writefile cleaning_pipeline.py

import pandas as pd


def clean_agent_data(txns, agents):
    """
    Clean agent transaction data.

    - Remove duplicate transactions.
    - Standardize transaction types.
    - Convert transaction amounts to numeric.
    - Standardize transaction dates.
    - Flag missing amounts and IQR outliers.
    - Merge transactions with agent details.
    - Preserve transactions belonging to unknown agents.

    Returns:
        clean_df: Cleaned transaction DataFrame.
        log: Dictionary containing cleaning statistics.
    """

    clean_df = txns.copy()
    agents_df = agents.copy()

    original_rows = len(clean_df)

    # 1. Remove duplicate transactions
    duplicates_removed = int(clean_df.duplicated().sum())
    clean_df = clean_df.drop_duplicates().copy()

    # 2. Standardize transaction types
    clean_df["txn_type"] = (
        clean_df["txn_type"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    # 3. Convert amounts to numeric
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"],
        errors="coerce"
    )

    clean_df["amount_missing"] = clean_df["amount"].isna()

    # Do not invent missing financial transaction amounts.
    # Keep them as NaN and flag them for investigation.

    # 4. Standardize transaction dates
    clean_df["txn_date"] = (
        clean_df["txn_date"]
        .astype("string")
        .str.replace("/", "-", regex=False)
    )

    clean_df["txn_date"] = pd.to_datetime(
        clean_df["txn_date"],
        errors="coerce"
    )

    # 5. Detect outliers using the IQR method
    valid_amounts = clean_df["amount"].dropna()

    if len(valid_amounts) > 0:
        q1 = valid_amounts.quantile(0.25)
        q3 = valid_amounts.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        clean_df["amount_outlier"] = (
            (clean_df["amount"] < lower_bound) |
            (clean_df["amount"] > upper_bound)
        )
    else:
        clean_df["amount_outlier"] = False

    # 6. Merge agent information while retaining all transactions
    clean_df = clean_df.merge(
        agents_df,
        on="agent_id",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    unknown_agents = int((clean_df["_merge"] == "left_only").sum())

    clean_df["county"] = clean_df["county"].fillna("Unknown")
    clean_df = clean_df.drop(columns=["_merge"])

    # 7. Cleaning log
    log = {
        "original_rows": original_rows,
        "duplicates_removed": duplicates_removed,
        "rows_after_deduplication": len(txns.drop_duplicates()),
        "final_rows": len(clean_df),
        "missing_amounts": int(clean_df["amount_missing"].sum()),
        "outliers_flagged": int(clean_df["amount_outlier"].sum()),
        "unknown_agent_transactions": unknown_agents,
        "invalid_dates": int(clean_df["txn_date"].isna().sum())
    }

    return clean_df, log

Writing cleaning_pipeline.py


In [48]:
import os
from cleaning_pipeline import clean_agent_data

print("File exists:", os.path.exists("cleaning_pipeline.py"))
print("Function imported:", callable(clean_agent_data))

File exists: True
Function imported: True


In [49]:
import os

files_to_check = [
    "agents.csv",
    "agent_transactions.csv",
    "generate_data.py"
]

for file in files_to_check:
    print(
        f"{'FOUND' if os.path.exists(file) else 'MISSING'}: {file}"
    )

FOUND: agents.csv
FOUND: agent_transactions.csv
FOUND: generate_data.py


In [52]:
from google.colab import files
files.download("agent_transactions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import os

required_files = [
    "generate_data.py",
    "duka_sales_exploration.ipynb",
    "excel/duka_branch_summary.xlsx",
    "outputs/premium_online_coast_capital.csv"
]

for file in required_files:
    print(
        "FOUND:" if os.path.exists(file) else "MISSING:",
        file
    )

MISSING: generate_data.py
MISSING: duka_sales_exploration.ipynb
MISSING: excel/duka_branch_summary.xlsx
MISSING: outputs/premium_online_coast_capital.csv
